# Chapter 1, Exercise 4: Diacritizations of علم checked with CAMeL Tools

> **How to use this notebook.** Open it in Google Colab (File > Upload notebook, or the Colab badge on the companion website), then run the cells from top to bottom. Runtime > Change runtime type lets you pick a GPU when one is recommended below. Everything else runs on the free CPU tier.

## The exercise

**Chapter 1, Exercise 4.** For the written form علم, list at least three valid diacritizations with transliterations and meanings, and explain why this is a problem for a text-to-speech system. Validate your list with an open-source Arabic morphological analyzer such as CAMeL Tools [51], report at least four analyses with the vocalized form, the Buckwalter transliteration, the lemma, the part of speech, and an English gloss, and say which analyses lead to different pronunciations for a synthesizer.

## Requirements

No GPU is required; the notebook runs on Colab's free CPU runtime.


<a href="https://colab.research.google.com/github/arabic-speech-book/arabic-speech-book.github.io/blob/main/docs/solutions/Chapter_01_Exercise_04.ipynb" target="_blank" rel="noopener"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

## 1. Install CAMeL Tools and its MSA morphology database

CAMeL Tools (Obeid et al., 2020) ships a morphological analyzer backed by the CALIMA-MSA database. The database is downloaded separately with `camel_data`; it is about 40 MB.

In [1]:
!pip install -q camel-tools
!camel_data -i morphology-db-msa-r13

No new packages will be installed.


## 2. The hand-written list, for reference

| Vocalized form | Transliteration | IPA | Meaning | Ambiguity layer |
|---|---|---|---|---|
| عِلْم | ʿilm | /ʕilm/ | knowledge, science | word-internal vowels |
| عَلَم | ʿalam | /ʕalam/ | flag | word-internal vowels |
| عَلِمَ | ʿalima | /ʕalima/ | he knew | word-internal vowels |
| عَلَّمَ | ʿallama | /ʕalːama/ | he taught (Form II, geminate) | word-internal vowels plus shadda |
| عُلِمَ | ʿulima | /ʕulima/ | it was known (passive) | word-internal vowels |

Section 1.5.2 gives the first four; the analyzer below will tell us whether it agrees and what else it finds.

In [2]:
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer
import pandas as pd

db = MorphologyDB.builtin_db('calima-msa-r13')
analyzer = Analyzer(db)

analyses = analyzer.analyze('علم')
print(f"CAMeL Tools returned {len(analyses)} analyses for علم")

rows = []
for a in analyses:
    rows.append({"diac (vocalized)": a["diac"], "Buckwalter": a["bw"], "lemma": a["lex"],
                 "POS": a["pos"], "gloss": a["gloss"], "root": a.get("root", ""),
                 "pattern": a.get("pattern", "")})
df = pd.DataFrame(rows).drop_duplicates()
pd.set_option("display.max_colwidth", 60)
df

CAMeL Tools returned 21 analyses for علم


,diac (vocalized),Buckwalter,lemma,POS,gloss,root,pattern
0,عِلْم,عِلْم/NOUN,عِلْم,noun,science;study_of,ع.ل.م,1ِ2ْ3
1,عِلْمَ,عِلْم/NOUN+َ/CASE_DEF_ACC,عِلْم,noun,science;study_of+[def.acc.],ع.ل.م,1ِ2ْ3َ
2,عِلْمٌ,عِلْم/NOUN+ٌ/CASE_INDEF_NOM,عِلْم,noun,science;study_of+[indef.nom.],ع.ل.م,1ِ2ْ3ٌ
3,عِلْمِ,عِلْم/NOUN+ِ/CASE_DEF_GEN,عِلْم,noun,science;study_of+[def.gen.],ع.ل.م,1ِ2ْ3ِ
4,عِلْمٍ,عِلْم/NOUN+ٍ/CASE_INDEF_GEN,عِلْم,noun,science;study_of+[indef.gen.],ع.ل.م,1ِ2ْ3ٍ
5,عِلْمُ,عِلْم/NOUN+ُ/CASE_DEF_NOM,عِلْم,noun,science;study_of+[def.nom.],ع.ل.م,1ِ2ْ3ُ
6,عَلِمَ,عَلِم/PV+َ/PVSUFF_SUBJ:3MS,عَلِم,verb,know;find_out+he;it_<verb>,ع.ل.م,1َ2ِ3َ
7,عَلَم,عَلَم/NOUN,عَلَم,noun,flag;banner;badge,ع.ل.م,1َ2َ3
8,عَلَمَ,عَلَم/NOUN+َ/CASE_DEF_ACC,عَلَم,noun,flag;banner;badge+[def.acc.],ع.ل.م,1َ2َ3َ
9,عَلَمٌ,عَلَم/NOUN+ٌ/CASE_INDEF_NOM,عَلَم,noun,flag;banner;badge+[indef.nom.],ع.ل.م,1َ2َ3ٌ


## 3. Group the analyses by pronunciation

Many of the 20-odd analyses differ only in the **case ending** (nominative -u, accusative -a, genitive -i, with or without tanwīn). In pausal pronunciation, the style used by most synthesizers at the end of a phrase, these endings are not spoken, so they collapse to the same sound sequence. The distinct *stems* are what change the synthesized word.

In [3]:
import re
DIAC_FINAL = re.compile(r"[\u064B-\u0650\u0652]$")   # strip a final short vowel / tanwin / sukun

def pausal_stem(diac):
    return DIAC_FINAL.sub("", diac)

df["pausal form"] = df["diac (vocalized)"].map(pausal_stem)
groups = (df.groupby("pausal form")
            .agg(lemmas=("lemma", lambda s: ", ".join(sorted(set(s)))),
                 pos=("POS", lambda s: ", ".join(sorted(set(s)))),
                 glosses=("gloss", lambda s: "; ".join(sorted(set(g.split('+')[0] for g in s)))),
                 n_analyses=("diac (vocalized)", "size"))
            .reset_index())
groups

,pausal form,lemmas,pos,glosses,n_analyses
0,عَلَم,عَلَم,noun,flag;banner;badge,6
1,عَلِم,عَلِم,verb,know;find_out,1
2,عَلَّم,عَلَّم,verb,teach;instruct,1
3,عُلِم,عَلِم,verb,be_known;be_found_out,1
4,عِلْم,عِلْم,noun,knowledge;knowing; science;study_of,12


## 4. What this means for a synthesizer

* The analyzer confirms the four readings of Section 1.5.2 (عِلْم, عَلَم, عَلِمَ, عَلَّمَ) and adds a fifth stem, the passive عُلِمَ 'it was known'.
* **Five different pronunciations** are possible for one undiacritized string: /ʕilm/, /ʕalam/, /ʕalima/, /ʕalːama/ (with a geminate /lː/, Section 2.5), and /ʕulima/. Each is a different word.
* The **case-ending analyses** (عِلْمُ, عِلْمَ, عِلْمِ, عِلْمٌ, ...) only matter when the synthesizer speaks full iʿrāb in connected speech; in pausal position they are pronounced identically to the bare stem. A front end must therefore decide (a) which stem, from context, and (b) whether to realize the case ending at all, which depends on phrase position (Table 9.1, waṣl and waqf).
* A text-to-speech system that guesses the wrong stem does not sound "slightly off"; it says a different word. This is why Chapter 9 places diacritization on the critical path of the Arabic front end.